# Lezione 11 - Agente-a-Agente (A2A) Protocollo


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Cos'è il protocollo A2A?

Il **protocollo Agent-to-Agent (A2A)** è uno standard aperto che permette agli agenti AI di comunicare,
scoprirsi a vicenda e collaborare — anche quando sono costruiti su framework diversi o ospitati
da servizi differenti.

Concetti chiave:

- **Scoperta** – Gli agenti pubblicano una *Scheda agente* che descrive le loro capacità, rendendo
  facile per altri agenti (o orchestratori) trovare lo specialista giusto per un compito.
- **Scambio di messaggi** – Gli agenti scambiano messaggi strutturati attraverso un protocollo comune, così
  una richiesta da un agente può essere compresa e soddisfatta da un altro indipendentemente dalla sua interna
  implementazione.
- **Ciclo di vita del task** – A2A definisce stati come *inviato*, *in lavorazione*, *completato* e
  *fallito*, fornendo all'orchestratore piena visibilità su come un compito delegato sta progredendo.

In questa lezione simuliamo la collaborazione in stile A2A collegando tre agenti di viaggio specializzati
in un flusso di lavoro in cui ogni agente contribuisce con la propria esperienza e passa i risultati al successivo.


## Creazione di agenti di viaggio specializzati


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Collaborazione multi-agente tramite flusso di lavoro

Connettiamo i tre agenti in un flusso di lavoro sequenziale che rispecchia il passaggio di messaggi A2A:

1. **CurrencyExchangeAgent** riceve la richiesta dell'utente e fornisce indicazioni sulla valuta.
2. **ActivityPlannerAgent** riceve il contesto arricchito e aggiunge raccomandazioni sulle attività.
3. **TravelManagerAgent** sintetizza entrambi gli input in un brief di viaggio finale.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendere A2A in Produzione

In un ambiente di produzione il protocollo A2A abilita potenti scenari tra servizi:

| Funzionalità | Descrizione |
|---|---|
| **Interoperabilità tra framework** | Un agente costruito con un framework può delegare compiti a un agente costruito con qualsiasi altro framework compatibile con A2A, permettendo una vera interoperabilità tra organizzazioni. |
| **Confini dei servizi** | Gli agenti possono risiedere in microservizi separati, regioni cloud o addirittura in organizzazioni diverse pur collaborando senza soluzione di continuità. |
| **Scoperta dinamica** | Un orchestratore può interrogare un registro delle Schede agente in fase di esecuzione per trovare lo specialista più adatto a un dato sotto-compito. |
| **Streaming e notifiche push** | A2A supporta i Server-Sent Events (SSE) per aggiornamenti di avanzamento in tempo reale e notifiche push per attività di lunga durata. |

Il flusso di lavoro che abbiamo costruito sopra è una versione semplificata, eseguita nello stesso processo, di questo modello. In una distribuzione reale
ogni agente esporrebbe un endpoint HTTP, pubblicherrebbe una Scheda agente e comunicherebbe
tramite il protocollo A2A JSON-RPC.


## Riepilogo

In questa lezione hai imparato:

1. **Cos'è il protocollo A2A** — uno standard aperto per la scoperta tra agenti, la messaggistica,
   e la gestione dei compiti.
2. **Come creare agenti specializzati** — un agente di cambio valuta, un agente pianificatore di attività, e un orchestratore Gestore Viaggi.
3. **Come collegare gli agenti in un flusso di lavoro** — usando `WorkflowBuilder` per modellare il passaggio sequenziale
   di messaggi tra agenti.
4. **Come A2A funziona in produzione** — abilitando la collaborazione tra framework e tra servizi con scoperta dinamica e aggiornamenti in streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Dichiarazione di non responsabilità:
Questo documento è stato tradotto mediante il servizio di traduzione automatica basato su intelligenza artificiale Co-op Translator (https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire l'accuratezza, si segnala che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella lingua d'origine deve essere considerato la fonte autorevole. Per informazioni critiche si raccomanda di avvalersi di una traduzione professionale effettuata da un traduttore umano. Non siamo responsabili per eventuali incomprensioni o interpretazioni errate derivanti dall'uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
